In [1]:
import os

# Strict single-threading
os.environ["OMP_NUM_THREADS"] = "1"
os.environ["MKL_NUM_THREADS"] = "1"
os.environ["OPENBLAS_NUM_THREADS"] = "1"
os.environ["VECLIB_MAXIMUM_THREADS"] = "1"
os.environ["NUMEXPR_NUM_THREADS"] = "1"
os.environ["ITK_GLOBAL_DEFAULT_NUMBER_OF_THREADS"] = "1"

import gc
import cv2
import torch
import pandas as pd
import numpy as np
import SimpleITK as sitk
import torch.nn.functional as F

from scipy.ndimage import gaussian_filter1d
from tqdm import tqdm


# Disable OpenCV multi-threading
cv2.setNumThreads(0)
torch.set_num_threads(1)


def get_informative_slice_range_fast(
    volume: np.ndarray,
    threshold_percentile: int = 25,
    smooth_sigma: float = 2.0
) -> tuple[int, int]:

    n_slices = volume.shape[0]

    if n_slices <= 2:
        return 0, n_slices

    # Variance per slice
    variances = volume.var(axis=(1, 2))

    # Mean intensity per slice
    mean_intensities = volume.mean(axis=(1, 2))

    # Spatial gradients
    _, dy, dx = np.gradient(volume)

    edge_strengths = np.mean(
        np.abs(dy) + np.abs(dx),
        axis=(1, 2)
    )

    def normalize(arr):
        denom = arr.max() - arr.min() + 1e-8
        return (arr - arr.min()) / denom

    scores = (
        0.3 * normalize(mean_intensities)
        + 0.3 * normalize(variances)
        + 0.4 * normalize(edge_strengths)
    )

    scores = gaussian_filter1d(
        scores,
        sigma=smooth_sigma
    )

    threshold = np.percentile(
        scores,
        threshold_percentile
    )

    valid = scores > threshold

    if not np.any(valid):
        return 0, n_slices

    first = int(np.argmax(valid))

    last = int(
        len(valid) -
        np.argmax(valid[::-1])
    )

    if last <= first:
        return 0, n_slices

    return first, last


def test_process_single_study(
    test_df: pd.DataFrame,
    test_series_df: pd.DataFrame,
    test_series_root: str,
    num_slices: int = 16,
    spatial_size: tuple = (128, 128),
):
    """
    Preprocess all studies in test_df for inference.

    Returns:
        list[dict]

    Each dictionary contains only:
        study_id
        volumes
        plane
        fluid
        fat

    No labels or label_source are included.
    """

    sitk.ProcessObject_SetGlobalWarningDisplay(False)

    plane_map = {
        "Sagittal": 0,
        "Coronal": 1,
        "Axial": 2
    }

    processed_studies = []

    # ---------------------------------------------------------
    # Process one study at a time
    # ---------------------------------------------------------
    for study_uid in tqdm(
        test_df["StudyInstanceUID"].unique(),
        desc="Processing test studies"
    ):

        # Get all series belonging to this study
        study_series_rows = test_series_df[
            test_series_df["StudyInstanceUID"] == study_uid
        ]

        if study_series_rows.empty:
            continue

        volumes = []
        planes = []
        fluids = []
        fats = []

        # -----------------------------------------------------
        # Process each series
        # -----------------------------------------------------
        for _, row in study_series_rows.iterrows():

            series_uid = row["SeriesInstanceUID"]

            series_path = os.path.join(
                test_series_root,
                study_uid,
                series_uid
            )

            if not os.path.exists(series_path):
                continue

            # ---------------------------------------------
            # Get DICOM files
            # ---------------------------------------------
            try:
                dicom_files = (
                    sitk.ImageSeriesReader
                    .GetGDCMSeriesFileNames(series_path)
                )
            except Exception:
                dicom_files = []

            if not dicom_files:
                continue

            # ---------------------------------------------
            # Read DICOM series
            # ---------------------------------------------
            reader = sitk.ImageSeriesReader()
            reader.SetFileNames(dicom_files)

            try:
                image = reader.Execute()
            except Exception:
                continue

            volume = sitk.GetArrayFromImage(
                image
            ).astype(np.float32)

            if volume.shape[0] == 0:
                continue

            # ---------------------------------------------
            # 1. Informative slice trimming
            # ---------------------------------------------
            first, last = get_informative_slice_range_fast(
                volume,
                threshold_percentile=25,
                smooth_sigma=2.0
            )

            volume = volume[first:last]

            if volume.shape[0] == 0:
                continue

            # ---------------------------------------------
            # 2. Intensity normalization
            # ---------------------------------------------
            p1, p99 = np.percentile(
                volume,
                (1, 99)
            )

            volume = np.clip(
                volume,
                p1,
                p99
            )

            v_min = volume.min()
            v_max = volume.max()

            if v_max > v_min:

                volume = (
                    volume - v_min
                ) / (
                    v_max - v_min
                )

            else:

                volume = np.zeros_like(
                    volume,
                    dtype=np.float32
                )

            # ---------------------------------------------
            # 3. Resize to model input
            # ---------------------------------------------
            tensor_vol = (
                torch.from_numpy(volume)
                .unsqueeze(0)
                .unsqueeze(0)
            )

            resized_tensor = F.interpolate(
                tensor_vol,
                size=(
                    num_slices,
                    spatial_size[0],
                    spatial_size[1]
                ),
                mode="trilinear",
                align_corners=False
            )

            # [1, 1, S, H, W]
            # -> [1, S, H, W]
            resized_tensor = (
                resized_tensor
                .squeeze(0)
                .to(torch.float16)
            )

            volumes.append(
                resized_tensor
            )

            # ---------------------------------------------
            # Metadata
            # ---------------------------------------------
            planes.append(
                plane_map.get(
                    row["Anatomical_Plane"],
                    0
                )
            )

            fluids.append(
                int(row["Fluid_Sensitive"])
            )

            fats.append(
                int(row["Fat_Suppression"])
            )

        # -----------------------------------------------------
        # No valid series
        # -----------------------------------------------------
        if not volumes:
            print(
                f"WARNING: No valid series for study {study_uid}"
            )
            continue

        # -----------------------------------------------------
        # Create inference payload
        # -----------------------------------------------------
        payload = {
            "study_id": study_uid,

            "volumes": torch.stack(
                volumes,
                dim=0
            ),

            "plane": torch.tensor(
                planes,
                dtype=torch.long
            ),

            "fluid": torch.tensor(
                fluids,
                dtype=torch.long
            ),

            "fat": torch.tensor(
                fats,
                dtype=torch.long
            ),
        }

        processed_studies.append(
            payload
        )

        # Explicit cleanup
        del volumes
        del planes
        del fluids
        del fats

        gc.collect()

    return processed_studies

In [2]:
import pandas as pd


test_df = pd.read_csv("/kaggle/input/competitions/rsna-knee-abnormality-detection/test.csv")
test_series_df = pd.read_csv("/kaggle/input/competitions/rsna-knee-abnormality-detection/test_series.csv")
test_series_root = "/kaggle/input/competitions/rsna-knee-abnormality-detection/test_series"


test_data = test_process_single_study(test_df=test_df,
                                      test_series_df=test_series_df,
                                      test_series_root=test_series_root)

Processing test studies: 100%|██████████| 3/3 [00:24<00:00,  8.30s/it]


In [3]:
import torch
import torch.nn as nn
from torchvision.models import densenet121


class DenseNetSeriesEncoder(nn.Module):
    def __init__(
        self,
        radimagenet_weights_path=None,
        proj_dim=512,
        dropout=0.2,
        num_heads=4,
        dense_chunk_size=16,
        freeze_backbone=True,
        freeze_channel_adapter=True,
        radimagenet_weights=None,
    ):
        super().__init__()

        self.proj_dim = proj_dim
        self.dense_chunk_size = dense_chunk_size
        self.freeze_backbone = freeze_backbone
        self.freeze_channel_adapter = freeze_channel_adapter

        # ============================================================
        # 1. DenseNet121 backbone
        # ============================================================
        base = densenet121(weights=None)

        self.features = base.features

        # ============================================================
        # 2. Load RadImageNet DenseNet121 weights
        # ============================================================
        
        if radimagenet_weights_path is not None:
            self._load_radimagenet_weights(
                radimagenet_weights_path
            )

        # ============================================================
        # 3. Grayscale MRI -> RGB adapter
        # ============================================================
        self.channel_adapter = nn.Conv2d(
            in_channels=1,
            out_channels=3,
            kernel_size=1,
            bias=False
        )

        # # Equivalent to grayscale -> RGB replication/averaging
        with torch.no_grad():
            self.channel_adapter.weight.fill_(1.0 / 3.0)

        # unfreeze only denseblock4
        # with torch.no_grad():
        #     self.channel_adapter.weight.fill_(1.0)

        # unfreeze only denseblock4

        # Freeze adapter if requested
        for param in self.channel_adapter.parameters():
            param.requires_grad = not freeze_channel_adapter

        # ============================================================
        # 4. Freeze DenseNet backbone (except denseblock4 and norm5)
        # ============================================================
        # for param in self.features.parameters():
        #     param.requires_grad = not freeze_backbone

        # # DenseNet121 output channels = 1024
        # self.feature_dim = 1024


        # unfreeze only denseblock4

        # ============================================================
        # Store configuration
        # ============================================================

        self.freeze_backbone = freeze_backbone

        # ============================================================
        # DenseNet backbone freezing
        #
        # freeze_backbone=True
        #     -> Entire DenseNet frozen
        #
        # freeze_backbone=False
        #     -> Only denseblock4 trainable
        # ============================================================

        # First freeze EVERYTHING
        for param in self.features.parameters():
            param.requires_grad = False

        # Only unfreeze denseblock4 when requested
        if not self.freeze_backbone:

            if hasattr(self.features, "denseblock4"):

                for param in self.features.denseblock4.parameters():
                    param.requires_grad = True

            else:
                raise AttributeError(
                    "DenseNet does not contain 'denseblock4'"
                )

        # DenseNet121 output channels = 1024
        self.feature_dim = 1024

        # unfreeze only denseblock4

        # ============================================================
        # 5. Projection
        # ============================================================
        # This remains trainable even when DenseNet is frozen.
        self.projection = nn.Sequential(
            nn.Linear(self.feature_dim, proj_dim),
            nn.LayerNorm(proj_dim),
            nn.GELU(),
            nn.Dropout(dropout),
        )

        # ============================================================
        # 6. Slice-level self attention
        # ============================================================
        self.slice_attn = nn.MultiheadAttention(
            embed_dim=proj_dim,
            num_heads=num_heads,
            dropout=dropout,
            batch_first=True
        )

        self.slice_query = nn.Parameter(
            torch.randn(1, 1, proj_dim) * 0.02
        )

        self.slice_norm = nn.LayerNorm(proj_dim)


    # unfreeze only denseblock4

    def forward_features(self, x):

        # ========================================================
        # Frozen DenseNet layers
        # ========================================================

        x = self.features.conv0(x)
        x = self.features.norm0(x)
        x = self.features.relu0(x)
        x = self.features.pool0(x)

        x = self.features.denseblock1(x)
        x = self.features.transition1(x)

        x = self.features.denseblock2(x)
        x = self.features.transition2(x)

        x = self.features.denseblock3(x)
        x = self.features.transition3(x)

        # ========================================================
        # DenseBlock4
        # ========================================================

        if self.freeze_backbone:

            x = self.features.denseblock4(x)

        else:

            x = checkpoint(
                self.features.denseblock4,
                x,
                use_reentrant=False
            )

        # ========================================================
        # Final frozen normalization
        # ========================================================

        x = self.features.norm5(x)

        return x

    # unfreeze only denseblock4


    # ========================================================
    # Load RadImageNet
    # ========================================================
    
    def _load_radimagenet_weights(self, path):

        print(
            f"\nLoading RadImageNet weights from:\n{path}"
        )

        raw = torch.load(
            path,
            map_location="cpu"
        )

        # ----------------------------------------------------
        # Extract state dict
        # ----------------------------------------------------

        if isinstance(raw, dict):

            ckpt_state = raw.get(
                "state_dict",
                raw
            )

        else:

            ckpt_state = raw

        target_state = self.features.state_dict()

        target_keys = set(
            target_state.keys()
        )

        remapped = {}

        # ----------------------------------------------------
        # Match checkpoint -> DenseNet features
        # ----------------------------------------------------

        for k, v in ckpt_state.items():

            k_clean = k.replace(
                "module.",
                ""
            )

            matched = None

            for target_key in target_keys:

                if k_clean.endswith(target_key):

                    matched = target_key
                    break

            if (
                matched is not None
                and target_state[matched].shape == v.shape
            ):

                remapped[matched] = v

        # ----------------------------------------------------
        # Load
        # ----------------------------------------------------

        missing, unexpected = (
            self.features.load_state_dict(
                remapped,
                strict=False
            )
        )

        print(
            f"[RadImageNet load] "
            f"matched={len(remapped)}/{len(target_keys)} | "
            f"missing={len(missing)} | "
            f"unexpected={len(unexpected)}"
        )

        # ----------------------------------------------------
        # Warning
        # ----------------------------------------------------

        if len(remapped) < len(target_keys) * 0.90:

            print(
                "\nWARNING:"
                " fewer than 90% of DenseNet weights matched."
            )

            print(
                "Sample checkpoint keys:"
            )

            print(
                list(ckpt_state.keys())[:5]
            )

            print(
                "Sample target keys:"
            )

            print(
                list(target_keys)[:5]
            )

        print("RadImageNet loading completed.\n")


    # ================================================================
    # Keep frozen DenseNet in eval mode
    # ================================================================

    # def train(self, mode=True):
    #         super().train(mode)
    
    #         if self.freeze_backbone:
    #             # Keep early layers in eval mode (frozen stats), but let denseblock4 train
    #             self.features.eval()
                
    #         return self

    # unfreeze only denseblock4

    def train(self, mode=True):

        # First put the entire model into the requested mode
        super().train(mode)

        # ========================================================
        # DenseNet backbone handling
        # ========================================================

        if self.freeze_backbone:

            # ----------------------------------------------------
            # COMPLETE BACKBONE FREEZE
            # ----------------------------------------------------
            #
            # Keep the entire DenseNet in eval mode.
            #
            # This is especially important for BatchNorm layers.
            # ----------------------------------------------------

            self.features.eval()

        else:

            # ----------------------------------------------------
            # ONLY denseblock4 IS TRAINABLE
            # ----------------------------------------------------
            #
            # Keep all other DenseNet layers in eval mode.
            # ----------------------------------------------------

            self.features.eval()

            # Put ONLY denseblock4 into training mode
            if hasattr(self.features, "denseblock4"):

                self.features.denseblock4.train(mode)

            else:
                raise AttributeError(
                    "DenseNet does not contain 'denseblock4'"
                )

        return self

    # unfreeze only denseblock4

    # ================================================================
    # Forward
    # ================================================================
    def forward(self, x):
        """
        x:
            [B, N, C, S, H, W]

        Example:
            [4, 13, 1, 32, 128, 128]

        Returns:
            [B, N, proj_dim]
        """

        B, N, C, S, H, W = x.shape

        # ------------------------------------------------------------
        # Flatten study/series/slice dimensions
        # [B, N, C, S, H, W] -> [B*N*S, C, H, W]
        # ------------------------------------------------------------
        x = x.permute(
            0, 1, 3, 2, 4, 5
        ).contiguous()

        x = x.view(
            B * N * S,
            C,
            H,
            W
        )

        # ------------------------------------------------------------
        # Frozen feature extraction
        # ------------------------------------------------------------

        # unfreeze only denseblock4

        # ============================================================
        # Channel adapter
        # ============================================================

        x = self.channel_adapter(x)

        # ============================================================
        # DenseNet feature extraction in chunks
        # ============================================================

        feature_chunks = []

        for chunk in torch.split(
            x,
            self.dense_chunk_size,
            dim=0
        ):

            # --------------------------------------------------------
            # Completely frozen DenseNet
            # --------------------------------------------------------

            if self.freeze_backbone:

                with torch.no_grad():

                    feat = self.forward_features(chunk)

            # --------------------------------------------------------
            # DenseBlock4 trainable
            # --------------------------------------------------------

            else:

                feat = self.forward_features(chunk)

            # --------------------------------------------------------
            # Global average pooling
            # --------------------------------------------------------

            feat = F.relu(
                feat,
                inplace=False
            )

            feat = F.adaptive_avg_pool2d(
                feat,
                (1, 1)
            )

            feat = torch.flatten(
                feat,
                start_dim=1
            )

            feature_chunks.append(feat)

        x = torch.cat(
            feature_chunks,
            dim=0
        )

        # unfreeze only denseblock4

        # if self.freeze_backbone and self.freeze_channel_adapter:

        #     with torch.no_grad():
        #         x = self.channel_adapter(x)

        #         feature_chunks = []
        #         for chunk in torch.split(
        #             x,
        #             self.dense_chunk_size,
        #             dim=0
        #         ):
        #             feat = self.features(chunk)
        #             feat = F.relu(feat, inplace=False)
        #             feat = F.adaptive_avg_pool2d(feat, (1, 1))
        #             feat = torch.flatten(feat, start_dim=1)
        #             feature_chunks.append(feat)

        #         x = torch.cat(feature_chunks, dim=0)

        # else:
        #     x = self.channel_adapter(x)

        #     feature_chunks = []
        #     for chunk in torch.split(
        #         x,
        #         self.dense_chunk_size,
        #         dim=0
        #     ):
        #         feat = self.features(chunk)
        #         feat = F.relu(feat, inplace=False)
        #         feat = F.adaptive_avg_pool2d(feat, (1, 1))
        #         feat = torch.flatten(feat, start_dim=1)
        #         feature_chunks.append(feat)

        #     x = torch.cat(feature_chunks, dim=0)

        # ------------------------------------------------------------
        # Project DenseNet features (Trainable)
        # ------------------------------------------------------------
        x = self.projection(x)

        # ------------------------------------------------------------
        # Restore: [B*N*S, proj_dim] -> [B, N, S, proj_dim]
        # ------------------------------------------------------------
        x = x.view(
            B,
            N,
            S,
            self.proj_dim
        )

        # ------------------------------------------------------------
        # Slice-level attention with Intermediate Pooling
        # [B, N, S, D] -> [B*N, S, D]
        # ------------------------------------------------------------
        x = x.view(
            B * N,
            S,
            self.proj_dim
        )

        # --- Intermediate Pooling: Reduce S slices (e.g., 32) down to S//2 (16) ---
        # F.avg_pool1d expects shape: [Batch, Channels, Length]
        x = x.permute(0, 2, 1)                  # [B*N, D, S]
        x = F.avg_pool1d(x, kernel_size=2, stride=2)  # [B*N, D, S//2]
        x = x.permute(0, 2, 1)                  # [B*N, S//2, D]

        query = self.slice_query.expand(
            B * N,
            -1,
            -1
        )

        attended, _ = self.slice_attn(
            query=query,
            key=x,
            value=x,
            need_weights=False
        )

        # ------------------------------------------------------------
        # Residual + normalization
        # ------------------------------------------------------------
        attended = self.slice_norm(
            attended + query
        )

        # ------------------------------------------------------------
        # One representation per series: [B*N, 1, D] -> [B, N, D]
        # ------------------------------------------------------------
        attended = attended.squeeze(1)

        attended = attended.view(
            B,
            N,
            self.proj_dim
        )

        return attended
    
# ============================================================
# Metadata Embedding
# ============================================================

class MetadataEmbedding(nn.Module):

    def __init__(
        self,
        plane_dim=16,
        fluid_dim=8,
        fat_dim=8
    ):
        super().__init__()

        self.plane_embedding = nn.Embedding(
            3,
            plane_dim
        )

        self.fluid_embedding = nn.Embedding(
            2,
            fluid_dim
        )

        self.fat_embedding = nn.Embedding(
            2,
            fat_dim
        )

    def forward(
        self,
        plane_idx,
        fluid_idx,
        fat_idx
    ):

        p = self.plane_embedding(
            plane_idx
        )

        f1 = self.fluid_embedding(
            fluid_idx
        )

        f2 = self.fat_embedding(
            fat_idx
        )

        return torch.cat(
            [
                p,
                f1,
                f2
            ],
            dim=-1
        )


# ============================================================
# Study-Level Attention Pool
# ============================================================

class AttentionPool(nn.Module):

    def __init__(
        self,
        dim=544,
        num_heads=4,
        dropout=0.1
    ):
        super().__init__()

        self.query = nn.Parameter(
            torch.randn(
                1,
                1,
                dim
            ) * 0.02
        )

        self.attn = nn.MultiheadAttention(
            embed_dim=dim,
            num_heads=num_heads,
            dropout=dropout,
            batch_first=True
        )

        self.norm1 = nn.LayerNorm(
            dim
        )

        self.ffn = nn.Sequential(

            nn.Linear(
                dim,
                dim
            ),

            nn.GELU(),

            nn.Dropout(
                dropout
            ),

            nn.Linear(
                dim,
                dim
            )
        )

        self.norm2 = nn.LayerNorm(
            dim
        )

    def forward(
        self,
        series_tokens,
        key_padding_mask=None
    ):

        B = series_tokens.size(0)

        q = self.query.expand(
            B,
            -1,
            -1
        )

        pooled, attn_weights = self.attn(
            q,
            series_tokens,
            series_tokens,
            key_padding_mask=key_padding_mask
        )

        pooled = self.norm1(
            pooled.squeeze(1)
        )

        pooled = pooled + self.ffn(
            pooled
        )

        pooled = self.norm2(
            pooled
        )

        return pooled, attn_weights


# ============================================================
# Complete Knee Study Model
# ============================================================

class KneeStudyModel(nn.Module):

    def __init__(
        self,
        num_classes=12,
        radimagenet_weights_path=None,
        series_dim=512,
        metadata_dim=32,
        num_heads=4,
        classifier_hidden=256,
        dropout=0.3,
        dense_chunk_size=16,
        freeze_backbone=True,
        freeze_channel_adapter=True,
    ):
        super().__init__()

        # ----------------------------------------------------
        # MRI encoder
        # ----------------------------------------------------

        # self.series_encoder = DenseNetSeriesEncoder(

        #     radimagenet_weights_path=
        #         radimagenet_weights_path,

        #     proj_dim=series_dim,

        #     dropout=0.2,

        #     num_heads=num_heads,

        #     dense_chunk_size=
        #         dense_chunk_size,

        #     freeze_backbone=
        #         freeze_backbone
        # )


        self.series_encoder = DenseNetSeriesEncoder(
            radimagenet_weights_path=radimagenet_weights_path,
            proj_dim=series_dim,
            dropout=0.2,
            num_heads=num_heads,
            dense_chunk_size=dense_chunk_size,
            freeze_backbone=freeze_backbone,
            freeze_channel_adapter=freeze_channel_adapter
        )
        # ----------------------------------------------------
        # Metadata encoder
        # ----------------------------------------------------

        self.metadata_encoder = MetadataEmbedding()

        # ----------------------------------------------------
        # MRI + metadata dimension
        # ----------------------------------------------------

        token_dim = (
            series_dim +
            metadata_dim
        )

        # 512 + 32 = 544

        # ----------------------------------------------------
        # Study-level attention
        # ----------------------------------------------------

        self.attn_pool = AttentionPool(

            dim=token_dim,

            num_heads=num_heads,

            dropout=0.1
        )

        # ----------------------------------------------------
        # Classifier
        # ----------------------------------------------------

        self.classifier = nn.Sequential(

            nn.Linear(
                token_dim,
                classifier_hidden
            ),

            nn.LayerNorm(
                classifier_hidden
            ),

            nn.GELU(),

            nn.Dropout(
                dropout
            ),

            nn.Linear(
                classifier_hidden,
                num_classes
            )
        )

    # ========================================================
    # Forward
    # ========================================================

    def forward(
        self,
        slices,
        plane,
        fluid,
        fat,
        mask=None
    ):

        # ----------------------------------------------------
        # MRI series features
        # ----------------------------------------------------

        series_feats = self.series_encoder(
            slices
        )

        # [B,N,512]

        # ----------------------------------------------------
        # Metadata
        # ----------------------------------------------------

        meta_feats = self.metadata_encoder(
            plane,
            fluid,
            fat
        )

        # [B,N,32]

        # ----------------------------------------------------
        # Combine
        # ----------------------------------------------------

        tokens = torch.cat(
            [
                series_feats,
                meta_feats
            ],
            dim=-1
        )

        # [B,N,544]

        # ----------------------------------------------------
        # Study-level aggregation
        #
        # IMPORTANT:
        #
        # mask:
        # True  = real series
        # False = padding
        #
        # MultiheadAttention expects:
        # True  = ignore
        # False = valid
        #
        # Therefore use ~mask.
        # ----------------------------------------------------

        if mask is not None:

            study_vector, attn_weights = (
                self.attn_pool(
                    tokens,
                    key_padding_mask=~mask
                )
            )

        else:

            study_vector, attn_weights = (
                self.attn_pool(
                    tokens
                )
            )

        # [B,544]

        # ----------------------------------------------------
        # Classification
        # ----------------------------------------------------

        logits = self.classifier(
            study_vector
        )

        # [B,12]

        return logits

In [4]:

DENSE_CHUNK_SIZE = 64 
FREEZE_BACKBONE = True 
FREEZE_CHANNEL_ADAPTER = True
RADIMAGENET_PATH = None


def build_model(
    device
):

    model = KneeStudyModel(

        num_classes=12,

        radimagenet_weights_path=
            RADIMAGENET_PATH,

        series_dim=512,

        metadata_dim=32,

        num_heads=4,

        classifier_hidden=256,

        dropout=0.3,

        dense_chunk_size=
            DENSE_CHUNK_SIZE,

        # IMPORTANT:
        # full fine-tuning
        # freeze_backbone=False

        # Freeze RadImageNet DenseNet121
        freeze_backbone=FREEZE_BACKBONE,
        freeze_channel_adapter=FREEZE_CHANNEL_ADAPTER,

    )

    model = model.to(
        device
    )

    return model



In [5]:
import torch

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# 1. Build the empty model architecture using your defined classes
model = build_model(device=device)

# 2. Load ONLY the state dictionary weights
checkpoint_path = "/kaggle/input/models/nitesh8527/knee-model/pytorch/default/7/best_model.pth"
state_dict = torch.load(checkpoint_path, weights_only=False, map_location=device)

# 3. Load the weights into the model
model.load_state_dict(state_dict)

model = model.to(device)
# model.eval()

In [6]:
# all_results = []

# label_columns = [
# "ACL",
# "MCL",
# "Medial Meniscus",
# "Lateral Meniscus",
# "Medial OA",
# "Lateral OA",
# "PF OA",
# "Effusion",
# "Synovitis",
# "Baker's",
# "Contusion",
# "Fracture",
# ]

# model.eval()

# with torch.inference_mode():

#     for sample in tqdm(
#         test_data,
#         desc="Running inference"
#     ):

#         study_id = sample["study_id"]

#         volumes = (
#             sample["volumes"]
#             .unsqueeze(0)
#             .to(device=device, dtype=torch.float32)
#         )

#         plane = (
#             sample["plane"]
#             .unsqueeze(0)
#             .to(device)
#         )

#         fluid = (
#             sample["fluid"]
#             .unsqueeze(0)
#             .to(device)
#         )

#         fat = (
#             sample["fat"]
#             .unsqueeze(0)
#             .to(device)
#         )

#         logits = model(
#             slices=volumes,
#             plane=plane,
#             fluid=fluid,
#             fat=fat
#         )

#         probabilities = torch.sigmoid(
#             logits
#         )[0].cpu().numpy()

#         result = {
#             "StudyInstanceUID": study_id
#         }

#         for label, probability in zip(
#             label_columns,
#             probabilities
#         ):
#             result[label] = float(probability)

#         all_results.append(result)

all_results = []

label_columns = [
    "ACL",
    "MCL",
    "Medial Meniscus",
    "Lateral Meniscus",
    "Medial OA",
    "Lateral OA",
    "PF OA",
    "Effusion",
    "Synovitis",
    "Baker's",
    "Contusion",
    "Fracture",
]

model.eval()

with torch.inference_mode():

    for sample in tqdm(
        test_data,
        desc="Running inference"
    ):

        study_id = sample["study_id"]

        volumes = (
            sample["volumes"]
            .unsqueeze(0)
            .to(device=device, dtype=torch.float32)
        )

        plane = (
            sample["plane"]
            .unsqueeze(0)
            .to(device=device)
        )

        fluid = (
            sample["fluid"]
            .unsqueeze(0)
            .to(device=device)
        )

        fat = (
            sample["fat"]
            .unsqueeze(0)
            .to(device=device)
        )

        logits = model(
            slices=volumes,
            plane=plane,
            fluid=fluid,
            fat=fat
        )

        probabilities = torch.sigmoid(logits)[0].cpu().numpy()

        result = {
            "StudyInstanceUID": study_id
        }

        for label, probability in zip(
            label_columns,
            probabilities
        ):
            result[label] = float(probability)

        all_results.append(result)

Running inference: 100%|██████████| 3/3 [00:01<00:00,  1.51it/s]


In [7]:
submission = pd.DataFrame(all_results)

In [8]:
submission

,StudyInstanceUID,ACL,MCL,Medial Meniscus,Lateral Meniscus,Medial OA,Lateral OA,PF OA,Effusion,Synovitis,Baker's,Contusion,Fracture
0,1.2.826.0.1.3680043.8.498.10047035057544427318...,0.133324,0.081049,0.209522,0.087372,0.129506,0.117233,0.219779,0.172041,0.176780,0.191067,0.133243,0.117832
1,1.2.826.0.1.3680043.8.498.10062861783145312629...,0.088517,0.061587,0.253794,0.099247,0.320597,0.275125,0.480866,0.151255,0.177047,0.303946,0.108801,0.151671
2,1.2.826.0.1.3680043.8.498.10067514707072572280...,0.129752,0.085243,0.208868,0.083085,0.111508,0.104849,0.211795,0.206975,0.176858,0.178118,0.127529,0.117644


In [9]:
submission.to_csv(
    "/kaggle/working/submission.csv",
    index=False,
    float_format="%.6f"
)